In [ ]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, mean_squared_error,
    mean_absolute_error, r2_score
)
print("✅ Imports done!")

In [ ]:
df = pd.read_csv("../data/cleaned_data.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
# Drop columns that would leak information into targets
drop_cols = [
    'Good_Investment',     # classification target
    'Future_Price',        # regression target
    'Price_in_Lakhs',      # direct price info — leakage
    'Price_per_SqFt',      # used to CREATE both targets — leakage
    'city_price_ratio',    # derived from Price_per_SqFt — leakage
    'below_city_median',   # derived from Price_per_SqFt — leakage
]

X = df.drop(columns=drop_cols)
y_class = df['Good_Investment']
y_reg   = df['Future_Price']

print("X shape:", X.shape)
print("\nFeatures used:", X.columns.tolist())
print("\ny_class distribution:\n", y_class.value_counts())
print(f"Positive class ratio: {y_class.mean():.2%}")
print("\ny_reg stats:\n", y_reg.describe())

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print("Classification - Train:", X_train_c.shape, "| Test:", X_test_c.shape)
print("Regression    - Train:", X_train_r.shape, "| Test:", X_test_r.shape)

In [ ]:
import shutil, os

# Clean broken mlflow logs if any
mlruns_path = "../mlflow_logs/mlruns"
if os.path.exists(mlruns_path):
    for item in os.listdir(mlruns_path):
        item_path = os.path.join(mlruns_path, item)
        meta = os.path.join(item_path, "meta.yaml")
        if os.path.isdir(item_path) and not os.path.exists(meta):
            shutil.rmtree(item_path)
            print(f"Removed broken experiment: {item}")

mlflow.set_tracking_uri("../mlflow_logs")
mlflow.set_experiment("Real_Estate_Advisor")
print("✅ MLflow setup done!")
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
clf_results = {}

# Class imbalance weight
pos_weight = (y_class == 0).sum() / (y_class == 1).sum()

# ── Random Forest Classifier ─────────────────────────────────
with mlflow.start_run(run_name="RandomForest_Classifier"):
    rf_clf = RandomForestClassifier(
        n_estimators=300, max_depth=15,
        min_samples_leaf=5, class_weight='balanced',
        random_state=42, n_jobs=-1
    )
    rf_clf.fit(X_train_c, y_train_c)
    y_pred_rf = rf_clf.predict(X_test_c)

    acc = accuracy_score(y_test_c, y_pred_rf)
    f1  = f1_score(y_test_c, y_pred_rf)

    mlflow.log_param("model", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 15)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    mlflow.sklearn.log_model(rf_clf, artifact_path="model")

    clf_results['RandomForest'] = {'model': rf_clf, 'accuracy': acc, 'f1': f1, 'preds': y_pred_rf}
    print(f"✅ Random Forest → Accuracy: {acc:.4f} | F1: {f1:.4f}")

# ── XGBoost Classifier ───────────────────────────────────────
with mlflow.start_run(run_name="XGBoost_Classifier"):
    xgb_clf = XGBClassifier(
        n_estimators=500, max_depth=7, learning_rate=0.03,
        subsample=0.85, colsample_bytree=0.85,
        min_child_weight=3, gamma=0.1,
        reg_alpha=0.1, reg_lambda=1.0,
        scale_pos_weight=pos_weight,
        random_state=42, eval_metric='logloss', verbosity=0
    )
    xgb_clf.fit(X_train_c, y_train_c,
                eval_set=[(X_test_c, y_test_c)], verbose=False)
    y_pred_xgb = xgb_clf.predict(X_test_c)

    acc = accuracy_score(y_test_c, y_pred_xgb)
    f1  = f1_score(y_test_c, y_pred_xgb)

    mlflow.log_param("model", "XGBClassifier")
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("learning_rate", 0.03)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    mlflow.sklearn.log_model(xgb_clf, artifact_path="model")

    clf_results['XGBoost'] = {'model': xgb_clf, 'accuracy': acc, 'f1': f1, 'preds': y_pred_xgb}
    print(f"✅ XGBoost      → Accuracy: {acc:.4f} | F1: {f1:.4f}")

In [ ]:
print("=== Classification Model Comparison ===\n")
for name, res in clf_results.items():
    print(f"{name:15} → Accuracy: {res['accuracy']:.4f} | F1: {res['f1']:.4f}")

best_clf_name = max(clf_results, key=lambda x: clf_results[x]['f1'])
best_clf      = clf_results[best_clf_name]['model']
best_clf_preds= clf_results[best_clf_name]['preds']
print(f"\n🏆 Best Classifier: {best_clf_name}")

In [ ]:
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test_c, best_clf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Good', 'Good'],
            yticklabels=['Not Good', 'Good'])
plt.title(f'Confusion Matrix — {best_clf_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test_c, best_clf_preds, target_names=['Not Good', 'Good']))

In [ ]:
reg_results = {}

# ── Random Forest Regressor ──────────────────────────────────
with mlflow.start_run(run_name="RandomForest_Regressor"):
    rf_reg = RandomForestRegressor(
        n_estimators=300, max_depth=15,
        min_samples_leaf=5, random_state=42, n_jobs=-1
    )
    rf_reg.fit(X_train_r, y_train_r)
    y_pred_rf_r = rf_reg.predict(X_test_r)

    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_rf_r))
    mae  = mean_absolute_error(y_test_r, y_pred_rf_r)
    r2   = r2_score(y_test_r, y_pred_rf_r)

    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(rf_reg, artifact_path="model")

    reg_results['RandomForest'] = {'model': rf_reg, 'rmse': rmse, 'mae': mae, 'r2': r2, 'preds': y_pred_rf_r}
    print(f"✅ Random Forest → RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

# ── XGBoost Regressor ────────────────────────────────────────
with mlflow.start_run(run_name="XGBoost_Regressor"):
    xgb_reg = XGBRegressor(
        n_estimators=500, max_depth=7, learning_rate=0.03,
        subsample=0.85, colsample_bytree=0.85,
        min_child_weight=3, random_state=42, verbosity=0
    )
    xgb_reg.fit(X_train_r, y_train_r,
                eval_set=[(X_test_r, y_test_r)], verbose=False)
    y_pred_xgb_r = xgb_reg.predict(X_test_r)

    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_xgb_r))
    mae  = mean_absolute_error(y_test_r, y_pred_xgb_r)
    r2   = r2_score(y_test_r, y_pred_xgb_r)

    mlflow.log_param("model", "XGBRegressor")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(xgb_reg, artifact_path="model")

    reg_results['XGBoost'] = {'model': xgb_reg, 'rmse': rmse, 'mae': mae, 'r2': r2, 'preds': y_pred_xgb_r}
    print(f"✅ XGBoost      → RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

In [ ]:
print("=== Regression Model Comparison ===\n")
for name, res in reg_results.items():
    print(f"{name:15} → RMSE: {res['rmse']:.4f} | MAE: {res['mae']:.4f} | R²: {res['r2']:.4f}")

best_reg_name = min(reg_results, key=lambda x: reg_results[x]['rmse'])
best_reg      = reg_results[best_reg_name]['model']
best_reg_preds= reg_results[best_reg_name]['preds']
print(f"\n🏆 Best Regressor: {best_reg_name}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test_r, best_reg_preds, alpha=0.3, color='purple', s=5)
plt.plot([y_test_r.min(), y_test_r.max()],
         [y_test_r.min(), y_test_r.max()],
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Price per SqFt')
plt.ylabel('Predicted Price per SqFt')
plt.title(f'Actual vs Predicted — {best_reg_name}')
plt.legend()
plt.tight_layout()
plt.savefig('../data/actual_vs_predicted.png')
plt.show()

In [ ]:
feat_imp = pd.Series(best_clf.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='bar', color='steelblue')
plt.title(f'Top 15 Feature Importances — {best_clf_name} Classifier')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/feature_importance_clf.png')
plt.show()

print("Top 15 features:")
print(feat_imp)

In [ ]:
joblib.dump(best_clf, "../models/classifier_model.pkl")
joblib.dump(best_reg, "../models/regressor_model.pkl")

feat_cols = X.columns.tolist()
joblib.dump(feat_cols, "../models/clf_feature_columns.pkl")
joblib.dump(feat_cols, "../models/reg_feature_columns.pkl")

print(f"✅ Best Classifier saved  → {best_clf_name}")
print(f"✅ Best Regressor saved   → {best_reg_name}")
print(f"✅ Feature columns saved  → {len(feat_cols)} features")

In [ ]:
import os
model_files = os.listdir("../models/")
print("Files in models/ folder:")
for f in sorted(model_files):
    print(f"  ✅ {f}")